## Generate Synthetic Streaming Credit Card Transaction Data



<div style="font-family:'Segoe UI',system-ui,-apple-system,sans-serif;background:linear-gradient(135deg,#1a0533 0%,#1e1145 30%,#0f172a 100%);border-radius:16px;padding:30px 24px 24px;color:#fff;position:relative;overflow:hidden;">
  <div style="position:absolute;bottom:-30%;left:-10%;width:400px;height:400px;background:radial-gradient(circle,rgba(168,85,247,0.1) 0%,transparent 70%);pointer-events:none;"></div>
  <div style="text-align:center;font-size:24px;font-weight:600;margin-bottom:20px;color:#e2e8f0;">Streaming Data Generation</div>
  <table style="width:100%;border-collapse:separate;border-spacing:12px 0;">
    <tr>
      <td style="width:30%;vertical-align:top;border-radius:12px;padding:16px;background:linear-gradient(180deg,rgba(147,51,234,0.18),rgba(147,51,234,0.04));border:1px solid rgba(168,85,247,0.2);">
        <div style="font-size:22px;margin-bottom:6px;">⚙️</div>
        <div style="font-size:13px;font-weight:700;color:#c084fc;margin-bottom:6px;">Configuration</div>
        <div style="font-size:10px;color:#a5b4c8;line-height:1.7;">▸ Kafka: topic, secrets, checkpoint<br/>▸ DataGen: users, merchants, rate<br/>▸ Lakebase: instance, database</div>
        <div style="display:inline-block;font-size:9px;padding:2px 8px;border-radius:10px;margin-top:8px;font-weight:600;background:rgba(147,51,234,0.15);color:#c084fc;">utils/config.py</div>
      </td>
      <td style="width:4%;vertical-align:middle;text-align:center;"><svg width="28" height="20" viewBox="0 0 28 20"><line x1="0" y1="10" x2="18" y2="10" stroke="#9333ea" stroke-width="2" stroke-dasharray="5 3" opacity="0.5"/><polygon points="16,4 28,10 16,16" fill="#ec4899" opacity="0.7"/></svg></td>
      <td style="width:30%;vertical-align:top;border-radius:12px;padding:16px;background:linear-gradient(180deg,rgba(236,72,153,0.18),rgba(236,72,153,0.04));border:1px solid rgba(244,114,182,0.2);">
        <div style="font-size:22px;margin-bottom:6px;">🔄</div>
        <div style="font-size:13px;font-weight:700;color:#f472b6;margin-bottom:6px;">dbldatagen Engine</div>
        <div style="font-size:10px;color:#a5b4c8;line-height:1.7;">▸ 10,000 unique users<br/>▸ 1,000 merchants<br/>▸ 1,000 rows/sec rate source</div>
        <div style="display:inline-block;font-size:9px;padding:2px 8px;border-radius:10px;margin-top:8px;font-weight:600;background:rgba(236,72,153,0.15);color:#f472b6;">Streaming Source</div>
      </td>
      <td style="width:4%;vertical-align:middle;text-align:center;"><svg width="28" height="20" viewBox="0 0 28 20"><line x1="0" y1="10" x2="18" y2="10" stroke="#ec4899" stroke-width="2" stroke-dasharray="5 3" opacity="0.5"/><polygon points="16,4 28,10 16,16" fill="#34d399" opacity="0.7"/></svg></td>
      <td style="width:30%;vertical-align:top;border-radius:12px;padding:16px;background:linear-gradient(180deg,rgba(16,185,129,0.18),rgba(16,185,129,0.04));border:1px solid rgba(52,211,153,0.2);">
        <div style="font-size:22px;margin-bottom:6px;">📨</div>
        <div style="font-size:13px;font-weight:700;color:#34d399;margin-bottom:6px;">Kafka Topic</div>
        <div style="font-size:10px;color:#a5b4c8;line-height:1.7;">▸ fraud_feature_eng_example<br/>▸ Continuous event stream<br/>▸ Consumed by 02_pipeline</div>
        <div style="display:inline-block;font-size:9px;padding:2px 8px;border-radius:10px;margin-top:8px;font-weight:600;background:rgba(16,185,129,0.15);color:#34d399;">SASL_SSL</div>
      </td>
    </tr>
  </table>
</div>



This notebook generates **synthetic streaming credit card transaction data** using the [dbldatagen](https://github.com/databrickslabs/dbldatagen) library and publishes it to a Kafka topic for downstream feature engineering.

The generated data simulates realistic credit card activity including varying transaction amounts, merchant categories, geographic locations, and device fingerprints — providing a rich feature space for fraud detection.

### Transaction Schema

| Field | Type | Description |
|---|---|---|
| `transaction_id` | STRING | Unique transaction identifier |
| `user_id` | STRING | Cardholder identifier |
| `merchant_id` | STRING | Merchant identifier |
| `amount` | DOUBLE | Transaction amount |
| `currency` | STRING | Currency code |
| `merchant_category` | STRING | Merchant category (e.g., grocery, electronics) |
| `payment_method` | STRING | Payment method (e.g., chip, contactless) |
| `ip_address` | STRING | Source IP address |
| `device_id` | STRING | Device fingerprint |
| `latitude` | DOUBLE | Transaction latitude |
| `longitude` | DOUBLE | Transaction longitude |
| `card_type` | STRING | Card type (e.g., credit, debit) |
| `timestamp` | TIMESTAMP | Event timestamp |

### Prerequisites

- Run `00_setup` first to create the Lakebase feature table
- Configure Kafka topic and credentials in `utils/config.py`


<!-- <img src="./images/streaming_fraud_data_generation.png" alt="Data Generation Architecture" width="800px" style="display: block; margin: 0 auto 20px auto"/> -->

## 1: Configure Spark for Streaming

Set up Spark configurations required for stateful streaming operations:
- **RocksDB state store** for efficient state management
- **Changelog checkpointing** for faster recovery
- **Shuffle partitions** tuned for throughput

In [0]:
dbutils.library.restartPython()
spark.conf.set("spark.sql.streaming.stateStore.providerClass", "com.databricks.sql.streaming.state.RocksDBStateStoreProvider")
spark.conf.set("spark.sql.shuffle.partitions", "4")


## 2: Import Libraries

Load PySpark functions, the `TransactionDataGenerator` utility, and pipeline configuration.

In [0]:
# Import required libraries
import logging
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Import utility modules
from utils.data_generator import TransactionDataGenerator
from utils.config import Config

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


## 3: Generate Streaming Transaction Data

Create a streaming source that continuously generates synthetic credit card transactions using `dbldatagen`. The generator is configured with:
- **num_users** — Number of unique cardholders
- **num_merchants** — Number of unique merchants
- **rows_per_second** — Streaming throughput rate

> 💡 The `TransactionDataGenerator` wraps `dbldatagen`'s rate source to produce realistic, schema-conformant transaction events. See `utils/data_generator.py` for the full implementation.

In [0]:
#Get Config
config = Config()

# Generate streaming transaction data
data_gen = TransactionDataGenerator(spark)
df_transactions = data_gen.generate_transaction_data(
    num_users=config.data_gen_config["num_users"],              # unique users
    num_merchants=config.data_gen_config["num_merchants"],      # unique merchants
    rows_per_second=config.data_gen_config["rows_per_second"]     # transactions per second
)


## 4: Write Stream Data to Kafka Topic


Publish the streaming DataFrame to a Kafka topic with:
- **SASL_SSL** authentication using Databricks secrets
- **JSON serialization** of all transaction fields
- **Checkpointing** for exactly-once delivery guarantees

> 💡 Databricks secrets must be configured before running this cell. See the [secrets documentation](https://docs.databricks.com/aws/en/security/secrets/#secrets-overview) for details.

<!-- <img src="./images/streaming_fraud_kafka_write.png" alt="Kafka Write Flow" width="800px" style="display: block; margin: 0 auto 20px auto"/> -->

In [0]:
#Get Kafka Config
kafka_credentials_secrets = config.kafka_config["kafka_credentials_secrets"]
scope = kafka_credentials_secrets["scope"]

# Retrieve secrets from Databricks secret scope.
# Note: Daatabricks secrets should be stored prior to using them and it's not covered in this example
# You can find more information about Databricks secrets here: https://docs.databricks.com/aws/en/security/secrets/#secrets-overview
KAFKA_USERNAME = dbutils.secrets.get(scope = scope, key = kafka_credentials_secrets["username"])
KAFKA_SECRET = dbutils.secrets.get(scope = scope, key = kafka_credentials_secrets["secret"])
KAFKA_SERVER = dbutils.secrets.get(scope = scope, key = kafka_credentials_secrets["server"])
KAFKA_TOPIC = config.kafka_config["kafka_topic"]

#Data Generator Configuration
CHECKPOINT_BASE_PATH = config.kafka_config["checkpoint_base_path"]
CHECKPOINT_LOCATION = f"{CHECKPOINT_BASE_PATH}/transaction-data-generator-checkpoint"
#Since we generate data using synthentic dbldatagen tool, we need to delete old checkpoint directory everytime we run the notebook to avoid any conflicts
dbutils.fs.rm(CHECKPOINT_LOCATION, True)

#Convert row to JSON String
json_df = df_transactions.select(to_json(struct(*[col(c) for c in df_transactions.columns])).alias("value"), col("transaction_id").alias("key"))

kafkaWriter = (
    json_df
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_SERVER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{KAFKA_USERNAME}' password='{KAFKA_SECRET}';")
    .option("kafka.ssl.endpoint.identification.algorithm", "https")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("topic", KAFKA_TOPIC)
    .option("failOnDataLoss", "true")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
)

#Start Kafka Producer
kafkaQuery = kafkaWriter.start()

print(f"""
[STREAMING] Kafka producer started successfully
  Query ID: {kafkaQuery.id}
  Query Name: {kafkaQuery.name}
  Status: {kafkaQuery.status}
  Kafka Topic: {KAFKA_TOPIC}
  Checkpoint: {CHECKPOINT_LOCATION}
  Serialization: JSON
""")


## 5: Stop Kafka Writer (Optional)

Uncomment the cell below to stop the Kafka producer stream when data generation is complete.

In [0]:
# Stop Kafka Writer
# if kafkaQuery.isActive:
#     kafkaQuery.stop()
#     logger.info("Streaming query stopped")

# logger.info("\nPipeline complete!")